In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Ricerca automatica del file fusion
print("Ricerca file nella directory corrente...")
fusion_file_path = None

# Cerca in tutte le cartelle e sottocartelle
for root, dirs, files in os.walk("."):
    for file in files:
        # Cerca un file csv che abbia 'fusion' nel nome
        if file.endswith(".csv") and ("fusion" in file.lower() or "sensor" in file.lower()):
            fusion_file_path = os.path.join(root, file)
            print(f"--> TROVATO FILE: {fusion_file_path}")
            break
    if fusion_file_path:
        break

if fusion_file_path is None:
    print("\nERRORE CRITICO: Non è stato trovato nessun file CSV con 'fusion' o 'sensor' nel nome.")
    print("Ecco i file che ci sono qui:")
    for root, dirs, files in os.walk("."):
        for file in files:
            if file.endswith(".csv"):
                print(os.path.join(root, file))
    # Interrompiamo qui se non c'è il file
    raise FileNotFoundError("File non trovato. Controlla il nome.")

# Caricamento e analisi
df = pd.read_csv(fusion_file_path).fillna(-110)

# Separazione Feature RSSI vs IMU
# Logica: RSSI hanno il trattino '-' (es. 12-1), IMU no.
feat_rssi = [c for c in df.columns if '-' in c and c not in ['room', 'artwork']]
feat_imu = [c for c in df.columns if c not in feat_rssi and c not in ['room', 'artwork', 'timestamp', 'index', 'Unnamed: 0']]

print(f"\nFeature RSSI rilevate: {len(feat_rssi)}")
print(f"Feature IMU rilevate: {len(feat_imu)}")
print(f"Esempi IMU: {feat_imu[:5]}")

if len(feat_imu) == 0:
    print("ATTENZIONE: Colonne IMU non trovate. Forse file sbagliato")
else:
    X_rssi = df[feat_rssi]
    X_fusion = df[feat_rssi + feat_imu]
    y = df['artwork']

    # Configurazione Modello
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    print("\nCalcolo accuratezza SOLO RSSI (attendere)...")
    score_rssi = cross_val_score(rf, X_rssi, y, cv=cv, scoring='accuracy').mean()

    print("Calcolo accuratezza FUSION (attendere)...")
    score_fusion = cross_val_score(rf, X_fusion, y, cv=cv, scoring='accuracy').mean()

    print("\n" + "="*40)
    print(f"Accuratezza SOLO RSSI: {score_rssi*100:.2f}%")
    print(f"Accuratezza FUSION:    {score_fusion*100:.2f}%")
    print("="*40)
    diff = (score_fusion - score_rssi) * 100
    print(f"DIFFERENZA: {diff:+.2f}%")